# 05a — Young vs. old classifier (Lung)

Can a model tell old cells from young ones, within each cell type, for mice it has never seen?
One logistic regression per cell type; mice are held out together (`StratifiedGroupKFold`).
All functions live in `src/aging.py`.

### 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")          # so "src" can be imported from the notebook folder

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from src import aging as ag    # shared functions: edit src/aging.py, not the notebooks

adata = sc.read_h5ad("../data/lung_processed.h5ad")

### 2. Cell types with enough cells and mice in both groups

In [ ]:
ok = ag.usable_cell_types(adata)
ok

### 3. Main report (30-M-2 excluded from myeloid types)

`auroc_mouse` is the headline; `acc_train` near 1.0 is expected and shows overfitting, not performance.

In [ ]:
report = pd.DataFrame({ct: ag.age_eval(adata, ct, **ag.main_kw(ct)) for ct in ok}).T
report = report.astype(float).round(3).sort_values("auroc_mouse", ascending=False)
report.to_csv("../figure/lung_age_report.csv")
report

### 4. Sensitivity checks: excluded mice, sex, ambient genes

In [ ]:
clean = ag.feature_mask(adata, "clean")
males = [m for m in adata.obs["mouse.id"].astype(str).unique() if "-M-" in m]

res = pd.DataFrame(index=ok)
res["logreg"] = [ag.age_auc(adata, ct, **ag.main_kw(ct)) for ct in ok]
res["with_excluded_mice"] = [ag.age_auc(adata, ct) for ct in ok]
res["males_only"] = [ag.age_auc(adata, ct, keep_mice=males, **ag.main_kw(ct)) for ct in ok]
res["no_ambient_genes"] = [ag.age_auc(adata, ct, features=clean, **ag.main_kw(ct)) for ct in ok]
res.round(3)

### 5. Control: age labels shuffled between mice (should be ~0.5)

In [ ]:
res["shuffled"] = [np.nanmean([ag.age_auc(adata, ct, shuffle_seed=s, **ag.main_kw(ct)) for s in range(5)])
                   for ct in ok]
res.round(3)

### 6. Comparison model: LightGBM

In [ ]:
from lightgbm import LGBMClassifier

def lgbm():
    return LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=15,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.5, verbose=-1)

res["lgbm"] = [ag.age_auc(adata, ct, make_model=lgbm, **ag.main_kw(ct)) for ct in ok]
res.round(3)

### 7. Batch check: train on some age groups, test on unseen ones

Age and processing batch are tangled in TMS. A cell type only counts as having a real age signal if `cross_age` stays well above 0.5. Values far below 0.5 mean the model learned batch-specific differences that point the wrong way on new batches.

In [ ]:
res["cross_age"] = [ag.cross_age_auc(adata, ct, **ag.main_kw(ct)) for ct in ok]
res["passes_batch_check"] = res["cross_age"] >= 0.7
res.round(3).sort_values("cross_age", ascending=False)

### 8. Save table and figure

In [ ]:
res.to_csv("../figure/lung_age_auroc.csv")

order = res.sort_values("cross_age").index
ax = res.loc[order, ["logreg", "cross_age", "shuffled"]].plot.barh(figsize=(6, 6))
ax.axvline(0.5, color="gray", linestyle="--")
ax.set_xlabel("AUROC")
ax.set_title("Lung: mouse-held-out vs. unseen-age-group test")
ax.figure.savefig("../figure/lung_age_auroc.png", dpi=200, bbox_inches="tight")

### Findings

- **The batch check changes the picture.** Fibroblasts (0.97) and adventitial cells (0.95) pass clearly, and type II pneumocytes (0.80) pass on only 125 cells.
- **Neutrophils pass (0.78) but are unreliable:** their shuffled control is 0.70, far above the expected 0.5.
- **Immune cell types fail.** Monocytes, T, NK, and B cells score 0.15–0.40 on unseen age groups, so their high mouse-held-out scores mostly reflect batch.
- **Per-mouse AUROC of 1.0 is not proof.** Several cell types reach it and then fail the batch check, because batch and age coincide.
- **Other checks:** sex hides the signal in B and NK cells (males only: 0.77 and 0.89), removing ambient genes changes AUROC by at most 0.07 and never rescues a cell type on the batch check, and LightGBM is never better than logistic regression.